In [0]:
# Install Libraries
%pip install mlflow mysql-connector-python
%pip install mlflow==2.21.3

In [0]:
# Import Libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow
import mlflow.spark
import pandas as pd
import numpy as np
import boto3
import io
from pyspark.sql.functions import col

In [0]:
# Load the Dataset
resultsDF = spark.read.csv('s3a://columbia-gr5069-main/raw/results.csv', header=True, inferSchema=True)

In [0]:
# Set up features and target
features = ["grid", "laps", "milliseconds", "fastestLapSpeed"]
target = 'positionOrder'

resultsDF = resultsDF.withColumn("grid", col("grid").cast("double"))
resultsDF = resultsDF.withColumn("laps", col("laps").cast("double"))
resultsDF = resultsDF.withColumn("milliseconds", col("milliseconds").cast("double"))
resultsDF = resultsDF.withColumn("fastestLapSpeed", col("fastestLapSpeed").cast("double"))

# Drop missing values
resultsDF = resultsDF.dropna(subset=features + [target])


# Cast 'milliseconds' and 'fastestLapSpeed' to double
resultsDF = resultsDF.withColumn("milliseconds", col("milliseconds").cast("double"))
resultsDF = resultsDF.withColumn("fastestLapSpeed", col("fastestLapSpeed").cast("double"))

# Assemble features
vecAssembler = VectorAssembler(inputCols=features, outputCol="features")
results_vecDF = vecAssembler.transform(resultsDF)

# Train-test split
trainDF, testDF = results_vecDF.randomSplit([0.8, 0.2], seed=42)

In [0]:
# Model 1: Linear Regression Model

# Train Linear Regression Model
with mlflow.start_run(run_name="LinearRegression_Spark"):
    
    lr = LinearRegression(featuresCol="features", labelCol=target)
    lrModel = lr.fit(trainDF)
    
    predLR = lrModel.transform(testDF)
    
    # Evaluate 4 metrics
    evaluator_rmse = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="rmse")
    evaluator_r2 = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="r2")
    evaluator_mae = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="mae")
    evaluator_mse = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="mse")
    
    rmse_lr = evaluator_rmse.evaluate(predLR)
    r2_lr = evaluator_r2.evaluate(predLR)
    mae_lr = evaluator_mae.evaluate(predLR)
    mse_lr = evaluator_mse.evaluate(predLR)
    
    # Log model and metrics to MLflow
    mlflow.spark.log_model(lrModel, "linear_regression_model")
    mlflow.log_metric("rmse", rmse_lr)
    mlflow.log_metric("r2", r2_lr)
    mlflow.log_metric("mae", mae_lr)
    mlflow.log_metric("mse", mse_lr)
    mlflow.log_param("model_type", "LinearRegression")
    
    # Save predictions CSV and log artifact
    predLR_final = predLR.select(*features, target, "prediction")
    lr_csv_path = "/tmp/lr_predictions.csv"
    predLR_final.toPandas().to_csv(lr_csv_path, index=False)
    mlflow.log_artifact(lr_csv_path)


In [0]:
# Model 2: Random Forest Model

# Train Random Forest Model
with mlflow.start_run(run_name="RandomForest_Spark"):
    
    rf = RandomForestRegressor(featuresCol="features", labelCol=target, numTrees=100, maxDepth=10)
    rfModel = rf.fit(trainDF)
    
    predRF = rfModel.transform(testDF)
    
    # Evaluate 4 metrics
    rmse_rf = evaluator_rmse.evaluate(predRF)
    r2_rf = evaluator_r2.evaluate(predRF)
    mae_rf = evaluator_mae.evaluate(predRF)
    mse_rf = evaluator_mse.evaluate(predRF)
    
    # Log model and metrics to MLflow
    mlflow.spark.log_model(rfModel, "random_forest_model")
    mlflow.log_metric("rmse", rmse_rf)
    mlflow.log_metric("r2", r2_rf)
    mlflow.log_metric("mae", mae_rf)
    mlflow.log_metric("mse", mse_rf)
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 10)
    
    # Save predictions CSV and log artifact
    predRF_final = predRF.select(*features, target, "prediction")
    rf_csv_path = "/tmp/rf_predictions.csv"
    predRF_final.toPandas().to_csv(rf_csv_path, index=False)
    mlflow.log_artifact(rf_csv_path)


In [0]:
# Save Linear Regression prediction
predLR_final.write.format('jdbc').options(
    url='jdbc:mysql://xw3003-gr5069.ccqalx6jsr2n.us-east-1.rds.amazonaws.com/gr5069',
    driver='com.mysql.jdbc.Driver',
    dbtable='lr_model_predictions',
    user='admin',
    password='ILUDalian3280#'
).mode('overwrite').save()

In [0]:
# Save Random Forest prediction
predRF_final.write.format('jdbc').options(
    url='jdbc:mysql://xw3003-gr5069.ccqalx6jsr2n.us-east-1.rds.amazonaws.com/gr5069',
    driver='com.mysql.jdbc.Driver',
    dbtable='rf_model_predictions',
    user='admin',
    password='ILUDalian3280#'
).mode('overwrite').save()


In [0]:
# View Linear Regression Table
spark.read.format("jdbc").option("url", "jdbc:mysql://xw3003-gr5069.ccqalx6jsr2n.us-east-1.rds.amazonaws.com/gr5069") \
.option("driver", "com.mysql.jdbc.Driver") \
.option("dbtable", "lr_model_predictions") \
.option("user", "admin") \
.option("password", "ILUDalian3280#") \
.load().display()

# View Random Forest Table
spark.read.format("jdbc").option("url", "jdbc:mysql://xw3003-gr5069.ccqalx6jsr2n.us-east-1.rds.amazonaws.com/gr5069") \
.option("driver", "com.mysql.jdbc.Driver") \
.option("dbtable", "rf_model_predictions") \
.option("user", "admin") \
.option("password", "ILUDalian3280#") \
.load().display()